# DISTANCE FROM OPTICAL CENTER NOTEBOOK!! :D

Notes: 

1. Remember to be in your cdte-tools conda virtual environment when running this! You will need some of the cdte-tools packages

2. the `%matplotlib qt` below is needed for interactive plotting later. It will make the plot open in a new window! To use this, you need PyQt6 installed. I am using my `cdte-tools` environment since we are using some cdte-tools functions I would suggest installing PyQt6 in your cdte-tools environment by doing `conda install -c conda-forge pyqt` in the terminal with cdte-tools-env activated.

In [1]:
%matplotlib qt

## Step 1: Create a CdTe flight data dictionary so that the data is all parsed by target

Once you are done, `flightdata.CDTE_DICT` will have data saved at the correct times for each target for each CdTe. These targets will be used later when looking at the flight data itself. The target options are:
 - 'Target Alpha Pre-Shift'
- 'Target Alpha'
- 'Target Echo'
- 'Target Charlie' 
- 'Target Alpha 2'

Note: CdTe5 does not have any good counts for Target Alpha Pre-Shift

In [21]:
from utils.io.cdtetargetdict import CdTeDataParsing

cdte_filelist = [
    '/Users/pet00184/FOXSI_Analysis/FOXSI5_Data_Analysis/DRAFT_CdTe_Data/DRAFT_foxsi5_cdte1_level3_version1.fits',
    '/Users/pet00184/FOXSI_Analysis/FOXSI5_Data_Analysis/DRAFT_CdTe_Data/DRAFT_foxsi5_cdte3_level3_version1.fits',
    '/Users/pet00184/FOXSI_Analysis/FOXSI5_Data_Analysis/DRAFT_CdTe_Data/DRAFT_foxsi5_cdte4_level3_version1.fits',
    '/Users/pet00184/FOXSI_Analysis/FOXSI5_Data_Analysis/DRAFT_CdTe_Data/DRAFT_foxsi5_cdte5_level3_version1.fits'
]

flightdata = CdTeDataParsing(cdte_filelist)

 # ***FROM HERE ON OUT, WE WILL BE REDOING THIS PROCESS FOR EACH CDTE***

 ## Step 2: Find the optical center for a given CdTe using the April 17th Poker X-Ray Alignment data

Run the following cells until you are happy with a centroid! The centroid will be in arcseconds, and is saved as alignment.centroid_x and alignment.centroid_y which we will call when plotting the flight data.

Notes: 
- These files are level 3 files that have gone through solar coordinate conversion (assuming that the center of the detector is at 0, 0 arcseconds). to do this, you must first process the raw files through path3 and then use compute_solar_coords in cdte_tools_py.solar_coords.solar_coords. THE FILES HAVE A LOT OF COUNTS SO I WOULD NOTTTTT SUGGEST DOING THIS YOURSELF IT TAKES A LONG TIME TO GET THROUGH PATH2. It may be easiest for me to try and share them on a google drive, since each one is about 5.5 GB. 
- The prepare_image function is what takes the longest, so I keep it in its own cell. This way you can mess around with the centroid calculation and plotting without redoing the big data histogram.


In [20]:
from astropy.table import Table
from astropy.io import fits
from utils.alignment.opticalcenter import FOXSIAlignmentImage

# CdTe data should be the X-Ray alignment file that has been through path3 and solar coordinate conversion. Make sure that it is in an astropy Table structure!
cdte_data = Table(fits.open('/Users/pet00184/FOXSI_Analysis/FOXSI5_Data_Analysis/Resctructuring!!!!/Data/XRayAlignment/combined_foxsi_cdte4_run1549_PATH3_solar_coords.fits')[1].data)

# remember to put the correct CdTe
alignment = FOXSIAlignmentImage(
    cdte_data=cdte_data,
    cdte="CdTe4",
)

# this part takes the longest, so we are separating the plotting and the preparation into two different cells.
alignment.prepare_image(
    qualityflag=True,
    bins=250,
    plot_range=(-1000, 1000),
)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 array([-1000.,  -992.,  -984.,  -976.,  -968.,  -960.,  -952.,  -944.,
         -936.,  -928.,  -920.,  -912.,  -904.,  -896.,  -888.,  -880.,
         -872.,  -864.,  -856.,  -848.,  -840.,  -832.,  -824.,  -816.,
         -808.,  -800.,  -792.,  -784.,  -776.,  -768.,  -760.,  -752.,
         -744.,  -736.,  -728.,  -720.,  -712.,  -704.,  -696.,  -688.,
         -680.,  -672.,  -664.,  -656.,  -648.,  -640.,  -632.,  -624.,
         -616.,  -608.,  -600.,  -592.,  -584.,  -576.,  -568.,  -560.,
         -552.,  -544.,  -536.,  -528.,  -520.,  -512.,  -504.,  -496.,
         -488.,  -480.,  -472.,  -464.,  -456.,  -448.,  -440.,  -432.,
         -424.,  -416.,  -408.,  -400.,  -392.,  -384.,  -376.,  -368.,
         -360.,  -352.,  -344.,

In [22]:
# actually calculating the centroid and plotting!

fig, ax, im = alignment.plot_with_centroid(
    sigma=2, # sigma for gaussian smoothing when calculating centroid
    threshold_fraction=0.5, # Only bins with a value of at least 50% of the maximum are included when calculating the centroid
    x_range=(-100, 100), # ranges in arcseconds where we want to look for the centroid. For this, we just find the weighted average in x and y separately, but could move to a 2D gaussian
    y_range=(0, 200),
    circle_radius=300, # radius in arcseconds of the circle. Circle is centered around the centroid, a nice check to see the rings line up!
    xrange=600, # plotting ranges in arcseconds, 600 should be good for all CdTe (CdTe3 maybe you can go less)
    yrange=600,
)

CdTe4 centroid: Solar-X = 19.24 arcsec, Solar-Y = 96.77 arcsec


## Step 3: Use this centroid to find the distance of our interesting data from the optical center!

I have two interactive plot options to do this. Both will give you the following: 
- distance in arcseconds from the detector center
- distance in arcseconds and arcmin from the optical center
    - this has distance in x, y as well as total radial distance
- azimuth angle from optical center, assuming 0 degrees is straight up, and angles move clockwise.

Take your pick of the following!
1. Manually draw a box around where you want the centroid to be calculated. This will then automatically calculate the centroid within that box. Each time you make a new box, the information will update. The information will print out in the terminal for each new box, but the offsets will only be saved for the final box. 
2. The old version of clicking a specific point and getting the same criteria is still working, and this time also gets the distance from the optical center.

In [23]:
### DRAWING BOXES VERSION

import importlib
import utils.visualization.interactiveplotting as iff
importlib.reload(iff)

interactive_centroid = iff.InteractiveCdTeFlightCentroid(
    cdtedict=flightdata.CDTE_DICT
)

fig, ax = interactive_centroid.interactive_flight_centroid(
    target='Target Alpha 2',
    cdtekey='CdTe4',
    optical_center=(alignment.centroid_x, alignment.centroid_y),
    bins=150,
    qualityflag=True,
    energy_range=None,
    sigma=2,
    threshold_fraction=0.5,
    xrange=600,
    yrange=600,
)


CdTe4 — Target Target Alpha 2
Intensity-weighted centroid: X = -364.07, Y = 154.77 arcsec
ROI: X = [-533.32, -110.96], Y = [-67.97, 289.27] arcsec
Raw events inside ROI: 4458
Thresholded weighted standard deviation: X = 17.75, Y = 16.31 arcsec
Offset from optical center: dX = -383.32, dY = 58.01 arcsec
Radial offset: 387.68 arcsec (6.461 arcmin)
Azimuth: 278.6° (0° = +Solar-Y; clockwise toward +Solar-X)


In [ ]:
### JUST POINT PLOTTING

import utils.visualization.interactiveplotting as iff

interactive_offsets = iff.InteractiveCdTeOpticalOffset(
    cdtedict=flightdata.CDTE_DICT
)

fig, ax = interactive_offsets.interactive_optical_offset(
    target='Target Alpha 2',
    cdtekey='CdTe4',
    optical_center=(alignment.centroid_x, alignment.centroid_y),
    bins=150,
    qualityflag=True,
    energy_range=None,
    xrange=600,
    yrange=600,
)


Target Alpha 2 — CdTe4
Clicked position: (-504.18, 285.55) arcsec
Distance from detector center: 579.43 arcsec
Optical-center ΔX: -523.43 arcsec
Optical-center ΔY: +188.78 arcsec
Radial distance from optical center: 556.43 arcsec (9.27 arcmin)
Azimuth from north: 289.83°

Target Alpha 2 — CdTe4
Clicked position: (-520.93, 231.59) arcsec
Distance from detector center: 570.09 arcsec
Optical-center ΔX: -540.17 arcsec
Optical-center ΔY: +134.83 arcsec
Radial distance from optical center: 556.74 arcsec (9.28 arcmin)
Azimuth from north: 284.01°
